In [19]:
!pip install rocketcea cantera

ERROR: Could not find a version that satisfies the requirement CVodesIntegrator (from versions: none)
ERROR: No matching distribution found for CVodesIntegrator


In [2]:
from pathlib import Path
import rocketcea.cea_obj

path = Path(rocketcea.cea_obj.__file__)

text = path.read_text()

old = 'set_py_cea_line(N," rocket %s  %s"%(eqfrStr,pc_str) + subar_str + pcope_str + eps_str )'
new = 'set_py_cea_line(N," rocket %s ions  %s"%(eqfrStr,pc_str) + subar_str + pcope_str + eps_str )'

if old not in text:
    print("Target line NOT found — nothing changed.")
else:
    # Make a backup first
    backup = path.with_suffix(".py.backup")
    backup.write_text(text)

    # Patch it
    path.write_text(text.replace(old, new, 1))

    print("Patched:", path)
    print("Backup:", backup)

USER_HOME_DIR=/root
Patched: /usr/local/lib/python3.12/dist-packages/rocketcea/cea_obj.py
Backup: /usr/local/lib/python3.12/dist-packages/rocketcea/cea_obj.py.backup


In [5]:
from rocketcea.cea_obj import CEA_Obj
from rocketcea import cea_obj

card_str = """
fuel MyCustomPropellant C 6 N 32 O 6 F 21 H 9  wt=100.0
h,kj/mol=3050  t(k)=298.15
"""

cea_obj.add_new_fuel("MyCustomPropellant", card_str)

ispObj = CEA_Obj(propName="MyCustomPropellant")

Pc = 1000.0
eps = 100.0

final_text = ispObj.get_full_cea_output(
    Pc=Pc,
    eps=eps
)

_, _, _, mw_exhaust, gamma = ispObj.get_IvacCstrTc_ChmMwGam(
    Pc=Pc,
    eps=eps
)

with open("propellant_products.txt", "w") as f:
    f.write(final_text)

print(f"Exhaust MW: {mw_exhaust}, Gamma: {gamma}")

Exhaust MW: 23.841082875653274, Gamma: 1.297918983665867


In [6]:
!cat /kaggle/working/propellant_products.txt


 *******************************************************************************

         NASA-GLENN CHEMICAL EQUILIBRIUM PROGRAM CEA, OCTOBER 18, 2002
                   BY  BONNIE MCBRIDE AND SANFORD GORDON
      REFS: NASA RP-1311, PART I, 1994 AND NASA RP-1311, PART II, 1996

 *******************************************************************************



 reac
  name MyCustomPropellant C 6 N 32 O 6 F 21 H 9  wt=100.0
  h,kj/mol=3050  t(k)=298.15
  
 prob case=RocketCEA,
  rocket equilibrium   p,psia=1000.000000,  supar=100.000000,
  
  
  
 output calories  transport
 end

 OPTIONS: TP=F  HP=F  SP=F  TV=F  UV=F  SV=F  DETN=F  SHOCK=F  REFL=F  INCD=F
 RKT=T  FROZ=F  EQL=T  IONS=F  SIUNIT=F  DEBUGF=F  SHKDBG=F  DETDBG=F  TRNSPT=T

 TRACE= 0.00E+00  S/R= 0.000000E+00  H/R= 0.000000E+00  U/R= 0.000000E+00

 Pc,BAR =    68.947304

 Pc/P =

 SUBSONIC AREA RATIOS =

 SUPERSONIC AREA RATIOS =   100.0000

 NFZ=  1  Mdot/Ac= 0.000000E+00  Ac/At= 0.000000E+00

    REACTANT          WT.F

In [17]:
%%bash

cat > propellant.yaml << 'EOF'
description: |
  Equilibrium model for C6N32O6F21H9 monopropellant.
  Custom reactant enthalpy from bond-additivity estimate.
  Product species imported from Cantera's nasa_gas.yaml database.

phases:
- name: gas
  thermo: ideal-gas

  elements: [C, N, O, F, H]

  species:
  - species: [MyCustomPropellant]

  - nasa_gas.yaml/species:
    - H
    - H2
    - O
    - O2
    - OH
    - H2O
    - N
    - N2
    - NO
    - F
    - F2
    - HF
    - C
    - CO
    - CO2
    - CN
    - HCN
    - CF
    - CF2
    - CF3
    - CF4
    - COF2
    - COF
    - FCN
    - NF

  kinetics: none

  state:
    T: 298.15 K
    P: 1 atm

species:

- name: MyCustomPropellant

  composition:
    C: 6
    N: 32
    O: 6
    F: 21
    H: 9

  thermo:
    model: constant-cp
    T0: 298.15 K

    # 2650 matches the CEA run. Use 2250 or 3050 for the low and high cases
    h0: 3050 kJ/mol

    # Placeholder, same 74 atoms as before
    s0: 1224.5588 J/mol/K  
EOF

In [8]:
import cantera as ct

species = ct.Species.list_from_file("nasa_gas.yaml")
names = {sp.name for sp in species}

wanted = [
    "H", "H2", "O", "O2", "OH", "H2O",
    "N", "N2", "NO",
    "F", "F2", "HF",
    "C", "CO", "CO2", "CN", "HCN"
]

for name in wanted:
    print(f"{name:5s}", "OK" if name in names else "MISSING")

H     OK
H2    OK
O     OK
O2    OK
OH    OK
H2O   OK
N     OK
N2    OK
NO    OK
F     OK
F2    OK
HF    OK
C     OK
CO    OK
CO2   OK
CN    OK
HCN   OK


In [ ]:
import sys
import contextlib
import cantera as ct


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, s):
        for st in self.streams:
            st.write(s)
        return len(s)

    def flush(self):
        for st in self.streams:
            st.flush()


OUT = "/kaggle/working/cantera_output.txt"

with open(OUT, "w") as f, contextlib.redirect_stdout(Tee(sys.stdout, f)):
    gas = ct.Solution("/kaggle/working/propellant.yaml")

    T0 = 298.15
    Pc = 1000.0 * 6894.757293168

    gas.TPX = T0, Pc, {"MyCustomPropellant": 1.0}
    print("Initial state")
    print(gas.report())

    gas.equilibrate(
        "TV",
        solver="vcs",
        max_steps=100000000,
        max_iter=1000000,
        estimate_equil=-1,
        log_level=1
    )

    print("Equilibrium state")
    print(gas.report(threshold=0.0))
    print("Chamber temperature K", gas.T)
    print("Chamber pressure Pa", gas.P)
    print("Mean molecular weight", gas.mean_molecular_weight)
    print("Enthalpy J per kg", gas.enthalpy_mass)
    print("Entropy J per kg per K", gas.entropy_mass)
    print("Mole fractions")
    for name, x in sorted(zip(gas.species_names, gas.X), key=lambda p: -p[1]):
        print(name, x)

print("Saved to", OUT)